# Multi-Hop Investigator Agent


In [ ]:
%%capture
# 1: Environment Setup
!pip install -U langgraph langchain_groq langchain_community tavily-python langchain_huggingface langchain-tavily faiss-cpu neo4j python-dotenv -q

In [ ]:
# 2: Imports and Warnings
import warnings
warnings.filterwarnings("ignore")

import asyncio, faiss, operator, os, sys, json, logging, contextlib, io, time
from datetime import datetime
from typing import Literal, TypedDict, Annotated, List, Dict
from google.colab import userdata

from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableConfig
from langchain_core.exceptions import OutputParserException
from langchain_core.callbacks import BaseCallbackHandler
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from neo4j import GraphDatabase

print("Setup Complete. Ready.")

Setup Complete. Ready.


In [ ]:
# 3: Secrets and Configuration
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
os.environ["NEO4J_URI"] = userdata.get('NEO4J_URI')
os.environ["NEO4J_USER"] = userdata.get('NEO4J_USER')
os.environ["NEO4J_PASSWORD"] = userdata.get('NEO4J_PASSWORD')

In [ ]:
 # 4: Initialize LLM & Search

# Initialize LLM
model_name = "openai/gpt-oss-120b"
llm = ChatGroq(model=model_name, temperature=0.0)

# Initialize Embeddings (Wrapped in a context manager to swallow stubborn prints)
embedding_model_name = "all-MiniLM-L6-v2"
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

search_tool = TavilySearch(max_results=2)


# 5: Databases
# Vector Store Setup (Semantic Memory)
sample_dim = len(embeddings.embed_query("init"))
index = faiss.IndexFlatL2(sample_dim)
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

# Knowledge Graph Setup (Neo4j)
neo4j_driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)

try:
    with neo4j_driver.session() as session:
        session.run("RETURN 1")
    print("Neo4j Connection Successful.")
except Exception as e:
    print(f"Neo4j Connection Failed: {e}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Neo4j Connection Successful.


In [ ]:
# 6: Observability, State & Helpers

# Configure logging to match Requirement 5 formatting
logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
logger = logging.getLogger("Stage6")

class Stage6Observability(BaseCallbackHandler):
    def on_tool_start(self, serialized: Dict, input_str: str, **kwargs):
        # Specific requirement: [SEARCH TRIGGERED] tag
        logger.info(f"[SEARCH TRIGGERED] Querying Tavily: {input_str}")

    def on_llm_error(self, error, **kwargs):
        logger.error(f"[LLM ERROR] {error}")

callback_handler = Stage6Observability()

class AgentState(TypedDict):
    original_query: str
    active_query: str                 # Contextualized/Episodic query
    chat_history: Annotated[list[BaseMessage], operator.add]
    sub_queries: List[str]            # Decomposed roadmap
    research_ledger: Dict[str, str]   # Working memory of validated facts
    current_sub_query: str            # Active question from the roadmap
    search_query: str                 # The actual string sent to tools
    current_retrievals: str           # The raw data bundle for grading
    retry_count: int                  # Track for the 3-retry limit
    final_answer: str                 # Final synthesized report
    seen_hashes: Annotated[set, operator.or_] # FOR DEDUPLICATION: Prevents log repetition

def extract_triples(text: str) -> list[dict]:
    """Uses LLM to extract structured triples from text."""
    prompt = f"Extract entities and relationships from: {text}. Output ONLY JSON array: [{{'s': '...', 'p': '...', 'o': '...'}}]"
    try:
        return (llm | JsonOutputParser()).invoke(prompt)
    except Exception as e:
        logger.error(f"Triple Extraction Error: {e}")
        return []

def contextualize_query(state: AgentState) -> str:
    """Coreference Resolution: Turns 'they' or 'it' into specific entities based on history."""
    if not state.get("chat_history"):
        return state["original_query"]

    context_prompt = f"""
    History: {state['chat_history']}
    Current: {state['original_query']}
    Rewrite the Current query to be standalone. If 'they' refers to a specific company in the history, use the company name.
    """
    return llm.invoke(context_prompt).content.strip()

print("Enhanced Observability, State, and Helpers Ready.")

Enhanced Observability, State, and Helpers Ready.


In [ ]:
# 7: Node Definitions
from datetime import datetime

def planner_node(state: AgentState) -> Command[Literal["router"]]:
    """Decomposes complex queries into sub-questions."""
    print("\n--- PLANNER: CONTEXTUALIZING & DECOMPOSING ---")
    active_query = contextualize_query(state)

    prompt = f"""
    Break this query into logical, sequential sub-questions.
    Query: {active_query}
    Output JSON ONLY: {{"sub_queries": ["Question 1", "Question 2"]}}
    """
    try:
        plan = (llm | JsonOutputParser()).invoke(prompt)
        sub_queries = plan.get("sub_queries", [active_query])
    except:
        sub_queries = [active_query]

    logger.info(f'Sub-query Generation: "Original query broken into: {sub_queries}."')
    return Command(update={"active_query": active_query, "sub_queries": sub_queries, "research_ledger": {}}, goto="router")

def router_node(state: AgentState) -> Command[Literal["hybrid_searcher", "generator"]]:
    """Determines next sub-query and pivots context with strict logical grounding."""
    pending = [q for q in state["sub_queries"] if q not in state["research_ledger"]]
    if not pending:
        print("\n--- ROUTER: ALL QUESTIONS ANSWERED ---")
        return Command(goto="generator")

    next_query = pending[0]
    if state["research_ledger"]:
        context = json.dumps(state["research_ledger"])
        pivot_prompt = f"""
        Prior Findings: {context}
        Current Sub-Query: {next_query}
        Task:
        1. Identify the 'Core Subject' for the Current Sub-Query.
        2. If Prior Findings prove the Core Subject is non-existent/invalid, set 'search_query' to 'ABORT_SEARCH'.
        3. Otherwise, rewrite the Sub-Query into a standalone search query.
        Output JSON ONLY: {{"entity": "...", "detail": "...", "search_query": "..."}}
        """
        try:
            p = (llm | JsonOutputParser()).invoke(pivot_prompt)
            search_target = p.get("search_query", next_query)
            if search_target == "ABORT_SEARCH":
                logger.info(f'The Pivot: "LOGICAL ABORT - {next_query} depends on a non-existent subject."')
            else:
                logger.info(f'The Pivot: "Identified {p.get("entity")} from first search; now searching for {p.get("detail")} regarding {p.get("entity")}."')
        except: search_target = next_query
    else: search_target = next_query

    return Command(update={"current_sub_query": next_query, "search_query": search_target, "retry_count": 0}, goto="hybrid_searcher")

def hybrid_searcher_node(state: AgentState, config: RunnableConfig) -> Command[Literal["grader"]]:
    """Combines LTM + Web Search with Differential Smart Logging."""
    if state["search_query"] == "ABORT_SEARCH":
        return Command(update={"current_retrievals": "LOGICAL_ABORT: Subject does not exist."}, goto="grader")

    print(f"\n--- HYBRID SEARCH: {state['current_sub_query']} ---")
    combined_data = ""

    # 1. FAISS Semantic Memory
    v_results = vector_store.similarity_search_with_relevance_scores(state["search_query"], k=1)
    if v_results and v_results[0][1] >= 0.5:
        combined_data += f"[SEMANTIC MEMORY]: {v_results[0][0].page_content}\n"

    # 2. Neo4j Knowledge Graph
    try:
        entities = (llm | JsonOutputParser()).invoke(f"Extract entities: {state['search_query']}")
        with neo4j_driver.session() as session:
            for entity in entities:
                res = session.run("MATCH (n {name:$name})-[r]->(m) RETURN n.name + ' -' + r.type + '-> '+ m.name AS rel", name=entity)
                for record in res:
                    combined_data += f"[GRAPH MEMORY]: {record['rel']}\n"
    except: pass

    # 3. Tavily Web Search
    results = search_tool.invoke(state["search_query"], config={"callbacks": [callback_handler]})
    web_data = "\n".join([f"Source: {r['url']}\nContent: {r['content']}" for r in results["results"]])
    combined_data += f"[WEB DATA]:\n{web_data}"

    # DEDUPLICATED LOGGING
    print("\n" + "="*25 + " SEARCH AUDIT (DISPLAY ONLY) " + "="*25)
    if "[SEMANTIC MEMORY]" in combined_data: print("[CACHED] Semantic Memory report retrieved.")
    if "[GRAPH MEMORY]" in combined_data: print(f"[GRAPH] Found {combined_data.count('[GRAPH MEMORY]')} verified relationships.")
    for r in results["results"]: print(f"  > {r['url']} | {r['content'][:110]}...")
    print("="*79 + "\n")

    return Command(update={"current_retrievals": combined_data}, goto="grader")

def grader_node(state: AgentState) -> Command[Literal["router", "hybrid_searcher"]]:
    """Evaluates CURRENT_RETRIEVALS for logical sufficiency and temporal accuracy."""
    print(f"--- GRADER: EVALUATING (Attempt {state['retry_count'] + 1}) ---")

    if "LOGICAL_ABORT" in state["current_retrievals"]:
        logger.info('Relevance Grading: "Score: 1.0/1.0 - Premise invalid, stopping search."')
        ledger = state["research_ledger"]
        ledger[state["current_sub_query"]] = "Not Applicable (False Premise)"
        return Command(update={"research_ledger": ledger}, goto="router")

    current_date = datetime.now().strftime("%B %Y")

    # THE REFINED ELEMENT EVALUATION PROMPT
    prompt = f"""
    Evaluate the CURRENT_RETRIEVALS against the SUB_QUERY for logical sufficiency.

    SUB_QUERY: {state['current_sub_query']}
    CURRENT_RETRIEVALS: {state['current_retrievals']}

    SCORING CRITERIA:
    1. RELEVANCE: Does the data address the core subject?
    2. SUFFICIENCY: Is there enough detail to provide a definitive answer?
    3. TEMPORAL ACCURACY: Does the data reflect the reality of {current_date}?

    Output JSON ONLY:
    {{
        "is_relevant": bool,
        "score": float,
        "answer": "Extracted fact if score > 0.8",
        "retry_query": "Optimized search if score < 0.8"
    }}
    """
    try:
        res = (llm | JsonOutputParser()).invoke(prompt)
    except:
        res = {"is_relevant": False, "score": 0.0, "retry_query": state["search_query"]}

    score = res.get("score", 0.0)

    if res.get("is_relevant") and score >= 0.8:
        logger.info(f'Relevance Grading: "Score: {score}/1.0 - Information sufficient. Saving to ledger."')
        ledger = state["research_ledger"]
        ledger[state["current_sub_query"]] = res.get("answer", "No answer extracted.")
        return Command(update={"research_ledger": ledger}, goto="router")
    else:
        logger.info(f'Relevance Grading: "Score: {score}/1.0 - Information insufficient. Triggering query rewrite..."')
        if state["retry_count"] < 2:
            return Command(update={"search_query": res.get("retry_query"), "retry_count": state["retry_count"] + 1}, goto="hybrid_searcher")
        else:
            ledger = state["research_ledger"]
            ledger[state["current_sub_query"]] = "Inconclusive results."
            return Command(update={"research_ledger": ledger}, goto="router")

def generator_node(state: AgentState) -> Command[Literal["memory_updater"]]:
    """Synthesizes the final report using abstract logical constraints."""
    print("\n--- GENERATOR: SYNTHESIZING FINAL REPORT ---")
    ledger_context = json.dumps(state["research_ledger"], indent=2)
    prompt = f"""
    You are a strict investigative researcher. Use ONLY the Research Ledger.
    Original Query: {state['active_query']}
    Research Ledger: {ledger_context}
    STRICT EPISTEMIC RULE:
    1. If a foundational premise is false, state it explicitly.
    2. Declare dependent query parts as "Not Applicable".
    3. FORBIDDEN from discussing speculative alternatives or "intended" subjects.
    """
    response = llm.invoke(prompt, config={"callbacks": [callback_handler]})
    return Command(update={"final_answer": response.content}, goto="memory_updater")

def memory_updater_node(state: AgentState) -> Command[Literal[END]]: # type: ignore
    """Commits final validated research into Long-Term Memory."""
    print("--- MEMORY UPDATER: COMMITTING TO LONG-TERM STORAGE ---")
    doc = Document(page_content=state['final_answer'], metadata={"query": state['active_query']})
    vector_store.add_documents([doc])
    triples = extract_triples(state.get("final_answer", ""))
    with neo4j_driver.session() as session:
        for t in triples:
            if isinstance(t, dict) and "s" in t and "o" in t and "p" in t:
                session.run("""
                    MERGE (s:Entity {name:$s}) MERGE (o:Entity {name:$o})
                    MERGE (s)-[r:RELATION {type:$p}]->(o) SET r.weight = coalesce(r.weight, 0) + 1
                """, s=str(t['s']), o=str(t['o']), p=str(t['p']))
    print("Double-Helix Sync Complete.")
    new_messages = [HumanMessage(content=state['original_query']), AIMessage(content=state['final_answer'])]
    return Command(update={"chat_history": new_messages}, goto=END)

In [ ]:
# 8: Graph Assembly
builder = StateGraph(AgentState)

builder.add_node("planner", planner_node)
builder.add_node("router", router_node)
builder.add_node("hybrid_searcher", hybrid_searcher_node)
builder.add_node("grader", grader_node)
builder.add_node("generator", generator_node)
builder.add_node("memory_updater", memory_updater_node)

builder.add_edge(START, "planner")
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
# 9: Test Case Execution
async def run_investigator():
    test_id = f"investigation_run_{int(time.time())}"
    config = {"configurable": {"thread_id": test_id}}

    print("\n" + "="*80)
    print("TURN 1: THE FIGMA/CANVA MULTI-HOP TEST")
    print("="*80)
    test_query = "Compare the primary revenue source of the company that acquired Figma (if applicable) with the primary revenue source of Canva."
    final_state = await graph.ainvoke({"original_query": test_query}, config)

    print("\nFINAL RESEARCH REPORT")
    print(final_state["final_answer"])

    print("\n" + "="*80)
    print("TURN 2: EPISODIC CONTEXT & NEO4J MEMORY RECALL TEST")
    print("="*80)
    print("Expectation: The Planner should resolve 'they' to Canva, and the Hybrid Searcher should extract triples/context directly from Neo4j/FAISS.")

    follow_up_query = "What is their primary revenue source based on our previous research?"
    follow_up_state = await graph.ainvoke({"original_query": follow_up_query}, config)

    print("\nFOLLOW-UP REPORT")
    print(follow_up_state["final_answer"])

if __name__ == "__main__":
    await run_investigator()


TURN 1: THE FIGMA/CANVA MULTI-HOP TEST

--- PLANNER: CONTEXTUALIZING & DECOMPOSING ---
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Sub-query Generation: "Original query broken into: ['Who acquired Figma, if any?', 'What is the primary revenue source of the acquiring company?', 'What is the primary revenue source of Canva?', 'How do the primary revenue sources of the acquiring company and Canva compare?']."

--- HYBRID SEARCH: Who acquired Figma, if any? ---
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
[SEARCH TRIGGERED] Querying Tavily: Who acquired Figma, if any?

========================= SEARCH AUDIT (DISPLAY ONLY) =========================
  > https://www.youtube.com/watch?v=KAFbkYeEFoA | One of the biggest tech acquisitions of all time just happened, Figma got bought by Adobe… for $20 BILLION! He...
  > https://finance.yahoo.com/news/adobe-failed-acquisition-figma-cost-151035766.html | # Adobe’s Fail

1. Addressing the RAG Triad Failures - In this task, the three core failure points of the RAG Triad (Retrieval, Augmentation and Generation) are systematically mitigated through a LangGraph workflow. Retrieval failures (pulling irrelevant data) are caught by the Grader Node that evaluates the CURRENT_RETRIEVALS before they can poison the context window. Augmentation failures (context stuffing) are prevented by the Planner Node,breaking complex queries into focused sub-queries, ensuring the LLM only processes one specific logical hop at a time. Finally, Generation failures (hallucinations) are neutralized by the Generator Node's strict epistemic constraints, which force the agent to explicitly declare false premises rather than inventing workarounds.
2. Balancing Efficiency vs. Accuracy - Agentic RAG inherently trades speed and API cost for precision. To prevent infinite loops and wastage, I implemented a strict early stopping mechanism. The Grader Node uses a relevance threshold of >= 0.8 to accept data. If the retrieval falls below this score, it triggers a query rewrite, but this self-correction loop is hard-capped at a maximum of three attempts (retry_count < 2). Furthermore, a logical "Circuit Breaker" instantly aborts searches for invalid premises (e.g., searching for Figma's non-existent acquirer), saving valuable time and compute.
3. LLM Grader vs. Vector Similarity - An LLM-based Grader is highly superior to simple FAISS vector similarity scores because vector (cosine) similarity only measures semantic closeness, not factual sufficiency or nuances of the exact reality. For instance, a vector search might return a 0.9 similarity score for a 2022 article stating "Adobe plans to acquire Figma." However, an LLM Grader possesses the cognitive reasoning to cross-reference that retrieved text against the current date (2026), realizing the information is factually insufficient and triggering a query rewrite.
4. Future Proofing - To integrate Agentic RAG with a local document store (like PDFs or Notion workspaces), the Hybrid Searcher would not require the Tavily web search tool. Rather, a sophisticated document retriever utilizing semantic chunking will be implemented. The Grader’s prompt would also pivot: instead of grading for "Temporal Accuracy" (web freshness), it would grade for "Document Lineage," ensuring every synthesized fact securely traces back to a specific page or paragraph within the proprietary local data where it is grounded against hallucinations.
